# Benchmarks — Latencia e Throughput

Mede o desempenho do pipeline em diferentes cenarios:

| Secao | Conteudo |
|-------|---------|
| 1 | Latencia de predicao individual (modelo) |
| 2 | Throughput batch (predicoes/segundo) |
| 3 | Benchmark do pipeline cognitivo completo |
| 4 | Benchmark do streaming processor |


## Setup


In [ ]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib

from src.features.build_features import build_features
from src.models.model_utils import split_features_target

df = build_features(
    input_csv='../data/raw/ecommerce_customer_churn_dataset.csv',
    output_csv='../data/processed/features.csv',
)
X_train, X_test, y_train, y_test = split_features_target(df)
rf_model = joblib.load('../models/rf_model.pkl')
print('Setup concluido')


## 1 — Latencia de Predicao Individual


In [ ]:
N_RUNS = 1000
single_row = X_test.iloc[[0]]

# Warm-up
for _ in range(10):
    rf_model.predict_proba(single_row)

times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    rf_model.predict_proba(single_row)
    times.append((time.perf_counter() - t0) * 1000)

times = np.array(times)
print(f'Predicao individual ({N_RUNS} runs):')
print(f'  Media:   {times.mean():.3f} ms')
print(f'  Mediana: {np.median(times):.3f} ms')
print(f'  P95:     {np.percentile(times, 95):.3f} ms')
print(f'  P99:     {np.percentile(times, 99):.3f} ms')
print(f'  Max:     {times.max():.3f} ms')


## 2 — Throughput Batch


In [ ]:
batch_sizes = [1, 10, 100, 500, 1000, 5000]
results = []

for bs in batch_sizes:
    batch = X_test.iloc[:bs]
    t0 = time.perf_counter()
    rf_model.predict_proba(batch)
    elapsed = time.perf_counter() - t0
    throughput = bs / elapsed
    results.append({'batch_size': bs, 'latency_ms': elapsed*1000, 'throughput_ps': throughput})
    print(f'Batch={bs:5d}  {elapsed*1000:7.2f}ms  {throughput:,.0f} pred/s')

df_bench = pd.DataFrame(results)
display(df_bench.round(2))


## 3 — Pipeline Cognitivo Completo


In [ ]:
from src.agents.analyst_agent import AnalystAgent
from src.agents.strategy_agent import StrategyAgent
from src.agents.auditor_agent import AuditorAgent
from src.llm.retriever import SimpleRetriever
from src.llm.generator import LLMGenerator
from src.llm.rag import ChurnRAG

analyst  = AnalystAgent()
strategy = StrategyAgent()
auditor  = AuditorAgent()
rag      = ChurnRAG(SimpleRetriever(), LLMGenerator())

sample = X_test.head(50)
scores = rf_model.predict_proba(sample)[:, 1]

N = len(sample)
t0 = time.perf_counter()
for i, (_, row) in enumerate(sample.iterrows()):
    feats = row.to_dict()
    score = float(scores[i])
    analysis  = analyst.run(score, feats, '')
    decision  = strategy.run(analysis, 'offer_discount')
    explanation = rag.run(f'u_{i}', score, feats)
    state = auditor.run({'churn_score': score, 'strategy': decision})

elapsed = time.perf_counter() - t0
print(f'Pipeline cognitivo em {N} usuarios:')
print(f'  Total:    {elapsed*1000:.1f} ms')
print(f'  Media:    {elapsed/N*1000:.2f} ms/usuario')
print(f'  Throughput: {N/elapsed:.0f} usuarios/s')


## 4 — Streaming Processor


In [ ]:
from src.streaming.event_processor import StreamProcessor, generate_synthetic_events

events = generate_synthetic_events(n=100, seed=0)
processor = StreamProcessor(model_path='../models/rf_model.pkl')

t0 = time.perf_counter()
results = processor.run(events, delay=0.0, verbose=False)
total_ms = (time.perf_counter() - t0) * 1000

summary = processor.summary()
print(f'Streaming benchmark ({len(events)} eventos):')
print(f'  Tempo total:     {total_ms:.1f} ms')
print(f'  Latencia media:  {summary["latency_mean_ms"]} ms/evento')
print(f'  Latencia max:    {summary["latency_max_ms"]} ms')
print(f'  Throughput:      {len(events)/total_ms*1000:.0f} eventos/s')
print(f'  Score medio:     {summary["score_mean"]}')
print(f'  Segmentos:       {summary["segments"]}')
